# Tira Machine Translation Demo

In [2]:
import sys
sys.version

'3.10.19 (main, Oct 21 2025, 16:43:05) [GCC 11.2.0]'

In [52]:
from datasets import load_from_disk
from IPython.display import Audio
from pathlib import Path
import os
import json
import soundfile as sf
from transformers import AutoModel, AutoTokenizer, pipeline
import torch
from tqdm import tqdm

In [4]:
dataset_dir = Path(os.environ['DATASETS'])
train_manifest = dataset_dir/'tira_asr_translated'/'nemo'/'nemo_train_manifest.jsonl'
val_manifest = dataset_dir/'tira_asr_translated'/'nemo'/'nemo_validation_manifest.jsonl'

model_dir = Path(os.environ['MODELS'])
mbart_path = model_dir/'mbart_ft_elan'

In [5]:
def load_json_lines(manifest_path):
    data = []
    with open(manifest_path) as f:
        lines = f.readlines()
    for line in lines:
        data.append(json.loads(line))
    return data


In [6]:
val_data = load_json_lines(val_manifest)
val_data[:2]

[{'audio_filepath': '/mnt/LocalStorage/mjsimmons//datasets/tira_asr_translated/nemo/validation/cfebfbf6-6692-42fc-9449-e64d4be49f7f.wav',
  'duration': 1.94,
  'text': 'the shepherd is good',
  'target_lang': 'en',
  'source_text': 'ìjɔ̀ kə̀cə̀lò',
  'source_lang': 'sw',
  'task': 'ast',
  'pnc': 'yes'},
 {'audio_filepath': '/mnt/LocalStorage/mjsimmons//datasets/tira_asr_translated/nemo/validation/67dc4b49-0289-43bd-b487-b265d87eba49.wav',
  'duration': 0.89,
  'text': 'he (the shepherd) is good',
  'target_lang': 'en',
  'source_text': 'kə̀cə̀lò',
  'source_lang': 'sw',
  'task': 'ast',
  'pnc': 'yes'}]

In [7]:
Audio(val_data[0]['audio_filepath'])

## Demo mBART
Text-based ideal baseline

In [8]:
mbart_pipeline = pipeline(
    task="translation",
    model=mbart_path,
    device=0,
    src_lang="sw_KE",
    tgt_lang="en_XX",
)

Device set to use cuda:0


In [38]:
# val_data[10]['source_text'], mbart_pipeline(val_data[10]['text']), val_data[10]['text']

def print_mbart_output(pipeline, data, i):
    print("mBART-50...")
    src = data[i]['source_text']
    reference = data[i]['text']
    pred = mbart_pipeline(src)[0]['translation_text']

    print("\n" + "="*40)
    print(f"REF:  {reference}")
    print(f"PRED: {pred}")
    print(f"SRC:  {src}")
    print("="*40)

In [39]:
i=60
print_mbart_output(mbart_pipeline, val_data, i)
ipd.Audio(val_data[i]['audio_filepath'])

mBART-50...

REF:  lions are tall
PRED: the lion is tall
SRC:  t̪ùlí t̪òrlà


## Demo Allophant
Switch notebooks!

## Demo Canary-1b
Automatic speech translation model

In [44]:
import json
import torch
import tempfile
import linecache
import IPython.display as ipd
from nemo.collections.asr.models import EncDecMultiTaskModel

def translate_manifest_row(model, manifest_path, index=0):
    """
    Reads a specific row from a manifest, runs inference, and compares with ground truth.
    
    Args:
        model: The loaded Canary 1B model.
        manifest_path (str): Path to the validation .jsonl file.
        index (int): The 0-based index of the row to process.
    """
    print("Nvidia Canary 1b...")
    manifest_path = str(manifest_path)
    
    # 1. Read the specific line using linecache (efficient for large files)
    # linecache indices are 1-based, so we add 1
    line = linecache.getline(manifest_path, index + 1)
    
    if not line:
        print(f"Error: Index {index} is out of bounds for file {manifest_path}")
        return

    # 2. Parse the JSON
    data = json.loads(line)
    audio_path = data.get('audio_filepath')
    reference = data.get('text', "N/A") # Or 'translation' depending on your file
    src = data.get('source_text', 'N/A')
    
    # 3. Create a temp manifest with JUST this one line
    # We write the original JSON object back to a temp file.
    # This preserves 'task', 'pnc', 'duration', etc. perfectly.
    with tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl', delete=True) as temp_manifest:
        temp_manifest.write(json.dumps(data) + "\n")
        temp_manifest.flush()
        
        # 4. Run Inference
        with torch.no_grad():
            translations = model.transcribe(
                audio=[temp_manifest.name], 
                batch_size=1,
                return_hypotheses=False
            )
    
    prediction = translations[0] if translations else ""

    # 5. Display Results
    print("\n" + "="*40)
    print(f"REF:  {reference}")
    print(f"PRED: {prediction.text}")
    print(f"SRC:  {src}")
    print("="*40)

In [ ]:
nemo_chkpt = "/home/mjsimmons/projects/tira_machine_translation_asru2025/nemo_experiments/tira_mt_asru2025/model.nemo"
print(f"Loading model from {nemo_chkpt}...")
model = EncDecMultiTaskModel.restore_from(nemo_chkpt, map_location='cuda')

# Switch to evaluation mode (crucial for disabling dropout/batchnorm updates)
model.eval()
print("Model loaded successfully!")

In [58]:
print(len(val_data))

843


In [ ]:
# i = 95: canary good
# i = 842: mBART good
# i = 592: approx translation both
# i = 213: mBART good
# i = 250: influence of other elicitation frames
# i = 118: canary good

i = 95
print(i)
print_mbart_output(mbart_pipeline, val_data, i)
translate_manifest_row(model, val_manifest, i)
ipd.Audio(val_data[i]['audio_filepath'])

In [50]:
# def get_model_outputs(data, manifest_path, index):
#     """
#     Reads a specific row from a manifest, runs inference, and compares with ground truth.
    
#     Args:
#         model: The loaded Canary 1B model.
#         manifest_path (str): Path to the validation .jsonl file.
#         index (int): The 0-based index of the row to process.
#     """
#     manifest_path = str(manifest_path)
    
#     # 1. Read the specific line using linecache (efficient for large files)
#     # linecache indices are 1-based, so we add 1
#     line = linecache.getline(manifest_path, index + 1)
    
#     if not line:
#         print(f"Error: Index {index} is out of bounds for file {manifest_path}")
#         return

#     # 2. Parse the JSON
#     data = json.loads(line)
#     audio_path = data.get('audio_filepath')
#     reference = data.get('text', "N/A") # Or 'translation' depending on your file
#     src = data.get('source_text', 'N/A')
    
#     # 3. Create a temp manifest with JUST this one line
#     # We write the original JSON object back to a temp file.
#     # This preserves 'task', 'pnc', 'duration', etc. perfectly.
#     with tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl', delete=True) as temp_manifest:
#         temp_manifest.write(json.dumps(data) + "\n")
#         temp_manifest.flush()
        
#         # 4. Run Inference
#         with torch.no_grad():
#             translations = model.transcribe(
#                 audio=[temp_manifest.name], 
#                 batch_size=1,
#                 return_hypotheses=False
#             )
    
#     canary_prediction = translations[0] if translations else ""
#     mbart_prediction = mbart_pipeline(src)[0]['translation_text']

#     return {
#         'src': src,
#         'ref': reference,
#         'mBART_hand_label_pred': mbart_prediction,
#         'canary_pred': canary_prediction,
#     }

    

In [ ]:
# rows = []
# for i, _ in enumerate(tqdm(val_data)):
#     rows.append(get_model_outputs(val_data, val_manifest, i))
# df = pd.DataFrame(rows)


In [56]:
# import pandas as pd
# df = pd.DataFrame(rows)
# df.head()

,src,ref,mBART_hand_label_pred,canary_pred
0,ìjɔ̀ kə̀cə̀lò,the shepherd is good,The shepherd is good.,"Hypothesis(score=0.0, y_sequence=tensor([1753,..."
1,kə̀cə̀lò,he (the shepherd) is good,The power is good.,"Hypothesis(score=0.0, y_sequence=tensor([1239,..."
2,àprí jə̀cə̀lò,the boy is good,The boy is good.,"Hypothesis(score=0.0, y_sequence=tensor([1203,..."
3,ðà áprí və́lɛ̀ðɛ̀,boy pulled it (sheep) (away from),boy pulled it (sheep) away,"Hypothesis(score=0.0, y_sequence=tensor([1753,..."
4,ðə̀ áprí və́lɛ̀ðɛ̀,boy pulled it (sheep) (away from),boy pulled it (sheep) away,"Hypothesis(score=0.0, y_sequence=tensor([1753,..."


In [57]:
# df.to_csv('canary_mbart_labels.csv')